# ML_Proj: Workload Prediction
This notebook is configured to run your updated, lightweight repository on Google Colab or Kaggle. It installs the necessary dependencies and runs the training scripts, which have been configured to finish in a few minutes.

In [ ]:
!git clone https://github.com/grass-overflow/ML_Proj.git
%cd ML_Proj

### Install Dependencies
Colab runs a very new Python and Numpy 2.x environment which breaks legacy ML code. 
Here we downgrade Numpy to a stable 1.x version, and use an environment variable to bypass a pip installation block on the deprecated `sklearn` package that `talos` incorrectly relies on.

In [ ]:
!pip install "numpy<2.0"
!SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL=True pip install talos==1.0.2

### Run the Training Scripts
You can run any of the training scripts below. **Make sure you have enabled a GPU in Colab (Runtime -> Change runtime type -> Hardware accelerator: T4 GPU).**

In [ ]:
# Train LSTM
!python lstm_training.py

In [ ]:
# Train HBNN (Bayesian Neural Network)
!python hbnn_training.py

In [ ]:
# Train LSTMD
!python lstmd_training.py

### Visualizations
The scripts above generate predictions and save them as `.csv` files in the `res/` directory. 
Run the cell below to plot the predictions versus the actual resource usage for your models!

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os

if not os.path.exists('res'):
    print("No results directory found. Please run a training script first.")
else:
    output_files = [f for f in glob.glob('res/output_*.csv') if 'train-' not in f]
    if len(output_files) == 0:
        print("No predictions found in the 'res' directory yet.")
    else:
        for file in output_files:
            df = pd.read_csv(file)
            if 'avgcpu' in df.columns and 'labelsavgcpu' in df.columns:
                plt.figure(figsize=(14, 5))
                plt.title(f"CPU Workload Prediction: {os.path.basename(file)}")
                plt.plot(df['labelsavgcpu'].values[:200], label='Actual CPU Usage', color='blue', alpha=0.6)
                plt.plot(df['avgcpu'].values[:200], label='Predicted CPU Usage', color='red', linestyle='--')
                plt.xlabel("Time Steps")
                plt.ylabel("Normalized CPU")
                plt.legend()
                plt.show()
                
                plt.figure(figsize=(14, 5))
                plt.title(f"Memory Workload Prediction: {os.path.basename(file)}")
                plt.plot(df['labelsavgmem'].values[:200], label='Actual Memory Usage', color='green', alpha=0.6)
                plt.plot(df['avgmem'].values[:200], label='Predicted Memory Usage', color='orange', linestyle='--')
                plt.xlabel("Time Steps")
                plt.ylabel("Normalized Memory")
                plt.legend()
                plt.show()
            elif 'labels' in df.columns:
                plt.figure(figsize=(14, 5))
                plt.title(f"Workload Prediction: {os.path.basename(file)}")
                plt.plot(df['labels'].values[:200], label='Actual Usage', color='blue', alpha=0.6)
                plt.plot(df[df.columns[1]].values[:200], label='Predicted Usage', color='red', linestyle='--')
                plt.xlabel("Time Steps")
                plt.ylabel("Normalized Usage")
                plt.legend()
                plt.show()